# CNN_5 Exp 8 : Test (7 ช่อง, Class-weighted Loss) seed 43

## 1. Import

In [ ]:
!pip install -q mne

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import resample_poly
import mne
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D

import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)

## 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
TRAIN_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
EVAL_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
TEST_DIR = TRAIN_DIR
SOURCE_DIR = {"test": TEST_DIR}
TRAIN_RESULT_DIR = PROJECT_DIR / "results" / "CNN_5_7ch_Exp8_class_weighted_seed43"
MODEL_PATH = TRAIN_RESULT_DIR / "cnn5_best_model.keras"
RESULT_DIR = TRAIN_RESULT_DIR / "test_results"

assert MODEL_PATH.exists(), f"ไม่พบโมเดล: {MODEL_PATH}"
assert (TEST_DIR / "test").exists(), f"ไม่พบโฟลเดอร์ Test: {TEST_DIR / 'test'}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 43
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

FS_RAW = 256
FS = 100
EPOCH_LEN_RAW = 30 * FS_RAW
EPOCH_LEN = 30 * FS
SEQ_LEN = 5
SEQ_SAMPLES = SEQ_LEN * EPOCH_LEN

ALL_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
SELECTED_CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
CHANNEL_IDX = [ALL_CHANNELS.index(c) for c in SELECTED_CHANNELS]
N_ALL_CHANNELS = len(ALL_CHANNELS)
N_CHANNELS = len(SELECTED_CHANNELS)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
SPLITS = ["test"]

BATCH_SIZE = 80
CHUNK = 1000

## 3. โหลดข้อมูล Test (เกณฑ์เดียวกับ Train)

In [ ]:
SKIPPED = {}

def load_split(split):
    files = sorted((SOURCE_DIR[split] / split).glob("*.npz"))
    SKIPPED[split] = []

    max_groups = 0
    for path in files:
        with np.load(path, allow_pickle=True) as data:
            max_groups += len(data["stage"])

    X = np.empty((max_groups, SEQ_SAMPLES, N_CHANNELS), dtype=np.float32)
    rows = []
    n = 0

    for path in files:
        subject = path.stem
        start_n = n
        start_rows = len(rows)

        with np.load(path, allow_pickle=True) as data:
            stages = data["stage"]
            n_valid = data["n_valid"]
            y_all = data["y"]
            epochs_all = data["epoch_indices"]
            x_all = data["X"]

            for g in range(len(stages)):
                stage = str(stages[g])

                if stage not in STAGES:
                    continue
                if int(n_valid[g]) != SEQ_LEN:
                    continue

                y_group = np.asarray(y_all[g]).astype(str).ravel()
                epochs = np.asarray(epochs_all[g]).astype(int).ravel()

                if len(y_group) != SEQ_LEN or not np.all(y_group == stage):
                    continue
                if len(epochs) != SEQ_LEN or not np.all(np.diff(epochs) == 1):
                    continue

                x = np.asarray(x_all[g], dtype=np.float32)

                if x.shape != (SEQ_LEN, EPOCH_LEN_RAW, N_ALL_CHANNELS):
                    continue
                if not np.isfinite(x).all():
                    continue

                x = x[:, :, CHANNEL_IDX]
                x = x.reshape(SEQ_LEN * EPOCH_LEN_RAW, N_CHANNELS)
                x = resample_poly(x, FS, FS_RAW, axis=0)

                X[n] = x
                n += 1
                rows.append({
                    "subject": subject,
                    "stage": stage,
                    "label": STAGES.index(stage),
                    "epoch_start": int(epochs[0]),
                    "epoch_end": int(epochs[-1]),
                })

        found_stages = set(row["stage"] for row in rows[start_rows:])
        if found_stages != set(STAGES):
            n = start_n
            del rows[start_rows:]
            SKIPPED[split].append(subject)

    if n == 0:
        raise ValueError(f"{split}: ไม่พบ Sequence ที่ผ่านเงื่อนไข")

    return X[:n], pd.DataFrame(rows)


DATA = {}
for split in SPLITS:
    X, meta = load_split(split)
    DATA[split] = {"X": X, "y": meta["label"].to_numpy().astype(np.int32), "meta": meta}

X_test, y_test = DATA["test"]["X"], DATA["test"]["y"]
meta_test = DATA["test"]["meta"]

assert X_test.shape[1:] == (SEQ_SAMPLES, N_CHANNELS), X_test.shape
assert set(np.unique(y_test)) <= set(LABELS)

## 4. ตรวจสอบข้อมูล Test (subject ไม่ซ้ำกับ Train/Val, จำนวน sequence)

In [ ]:
def subject_names(folder):
    return set(p.stem for p in folder.glob("*.npz"))

test_files = subject_names(TEST_DIR / "test")
test_subjects = set(meta_test["subject"])

other_subjects = {
    "multi5 train": subject_names(TRAIN_DIR / "train"),
    "multi5 val": subject_names(TRAIN_DIR / "val"),
    "preprocess train": subject_names(EVAL_DIR / "train"),
    "preprocess val": subject_names(EVAL_DIR / "val"),
}

for name, names in other_subjects.items():
    overlap = test_files & names
    print(f"{name}: {len(names)} subjects | ซ้ำกับ Test: {len(overlap)}")
    assert len(overlap) == 0, sorted(overlap)

print("\nไฟล์ Test ทั้งหมด:", len(test_files), "subjects")
print("subject ที่ถูกตัดออก (มีไม่ครบ 5 ระยะ):", SKIPPED["test"])

print("\n===== TEST DATA =====")
print("Test subjects:", len(test_subjects))
print("Test sequences:", len(meta_test))
print("X shape:", X_test.shape)

print("\nจำนวน Sequence ต่อคลาส")
print(meta_test["stage"].value_counts().reindex(STAGES, fill_value=0).to_string())

print("\nจำนวน Sequence ต่อ Subject และคลาส")
display(meta_test.groupby(["subject", "stage"]).size().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0))

print("\nmeta hash (ต้องเท่ากันทั้ง 3 Seeds):", int(pd.util.hash_pandas_object(meta_test, index=False).sum()))
meta_test.to_csv(RESULT_DIR / "test_sequences.csv", index=False)

## 5. โหลดโมเดลที่ Train เสร็จแล้ว (best checkpoint)

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()

## 6. ฟังก์ชันคำนวณ metric

In [ ]:
def calc_metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    overall = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision.mean(),
        "macro_recall": recall.mean(),
        "macro_f1": f1.mean(),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred, labels=LABELS),
    }
    per_class = pd.DataFrame({
        "stage": STAGES,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "support": support,
    })
    return overall, per_class


def stage_confusion(cm):
    rows = []
    total = cm.sum()
    for i, stage in enumerate(STAGES):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = total - tp - fp - fn
        rows.append({
            "stage": stage,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "sensitivity": tp / (tp + fn) if tp + fn > 0 else 0,
            "specificity": tn / (tn + fp) if tn + fp > 0 else 0,
        })
    return pd.DataFrame(rows)


def class_analysis(cm):
    rows = []
    for i, stage in enumerate(STAGES):
        errors = cm[i].copy()
        errors[i] = 0
        j = errors.argmax()
        rows.append({
            "stage": stage,
            "correct": int(cm[i, i]),
            "total": int(cm[i].sum()),
            "recall": cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0,
            "most_confused_with": STAGES[j] if errors[j] > 0 else "-",
            "confused_count": int(errors[j]),
        })
    return pd.DataFrame(rows)


def make_prediction_table(split, prob):
    meta = DATA[split]["meta"]
    pred = prob.argmax(axis=1)
    table = meta[["subject", "epoch_start", "epoch_end"]].copy()
    table["actual"] = [STAGES[i] for i in meta["label"]]
    table["predicted"] = [STAGES[i] for i in pred]
    table["confidence"] = prob.max(axis=1)
    for i, stage in enumerate(STAGES):
        table["prob_" + stage] = prob[:, i]
    table["correct"] = table["actual"] == table["predicted"]
    return table

## 7. ประเมินผล Test

In [ ]:
prob = np.concatenate([
    model.predict(X_test[i:i + CHUNK], batch_size=BATCH_SIZE, verbose=0)
    for i in range(0, len(X_test), CHUNK)
])
y_pred = prob.argmax(axis=1)

overall, per_class = calc_metrics(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred, labels=LABELS)
cm_norm = confusion_matrix(y_test, y_pred, labels=LABELS, normalize="true")
test_table = make_prediction_table("test", prob)

print("===== TEST: seed 43 =====")
display(pd.DataFrame([overall]).round(4))

print("Per-class metrics")
display(per_class.round(4))

print("Classification report")
report_text = classification_report(
    y_test, y_pred, labels=LABELS, target_names=STAGES, digits=4, zero_division=0
)
print(report_text)

print("Per-stage confusion (one-vs-rest)")
display(stage_confusion(cm).round(4))

print("Per-class analysis")
display(class_analysis(cm).round(4))

per_subject = test_table.groupby("subject").agg(
    sequences=("correct", "size"),
    correct=("correct", "sum"),
    accuracy=("correct", "mean"),
)
print("Accuracy ราย subject")
display(per_subject.round(4))

print("correct:", int(test_table["correct"].sum()), "/", len(test_table))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (count)")
ConfusionMatrixDisplay(cm_norm, display_labels=STAGES).plot(ax=axes[1], cmap="Blues", values_format=".2f", colorbar=False)
axes[1].set_title("Confusion matrix (normalized by actual)")
plt.suptitle("test | seed 43")
plt.tight_layout()
plt.savefig(RESULT_DIR / "test_confusion_matrix.png", dpi=150)
plt.show()

x_pos = np.arange(len(STAGES))
plt.figure(figsize=(8, 4.5))
plt.bar(x_pos - 0.27, per_class["precision"], 0.27, label="Precision")
plt.bar(x_pos, per_class["recall"], 0.27, label="Recall")
plt.bar(x_pos + 0.27, per_class["f1_score"], 0.27, label="F1-score")
plt.xticks(x_pos, STAGES)
plt.ylim(0, 1.05)
plt.title("test | seed 43: metrics per stage")
plt.legend()
plt.grid(axis="y", alpha=0.4)
plt.tight_layout()
plt.savefig(RESULT_DIR / "test_per_class_metrics.png", dpi=150)
plt.show()

pd.DataFrame([overall]).to_csv(RESULT_DIR / "test_overall_metrics.csv", index=False)
per_class.to_csv(RESULT_DIR / "test_per_class_metrics.csv", index=False)
stage_confusion(cm).to_csv(RESULT_DIR / "test_per_stage_confusion.csv", index=False)
class_analysis(cm).to_csv(RESULT_DIR / "test_class_analysis.csv", index=False)
pd.DataFrame(cm, index=STAGES, columns=STAGES).to_csv(RESULT_DIR / "test_confusion_matrix.csv")
per_subject.to_csv(RESULT_DIR / "test_per_subject_accuracy.csv")
test_table.to_csv(RESULT_DIR / "test_predictions.csv", index=False)
with open(RESULT_DIR / "test_classification_report.txt", "w") as f:
    f.write(report_text)

## 8. ตารางผลทำนายราย sequence (Test)

In [ ]:
print("correct:", int(test_table["correct"].sum()), "/", len(test_table))
display(test_table.round(3))

## 9. Hypnogram เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
print(f"Accuracy: {test_table['correct'].mean():.4f}")
print(f"Correct: {test_table['correct'].sum()}/{len(test_table)}")

stage_y = {"W": 4, "R": 3, "N1": 2, "N2": 1, "N3": 0}

hyp_dir = RESULT_DIR / "hypnogram"
hyp_dir.mkdir(parents=True, exist_ok=True)

FULL_NIGHT_DIR = PROJECT_DIR / "dataset" / "Test_Full_night"
EPOCH_SEC = 30

FIG_W, FIG_H = 30, 10
GT_COLOR = "#1f5fff"
PRED_LINE_COLOR = "black"
LINE_W = 1.5
BLOCK_W = 8
MARKER_SIZE = 90
PRED_MARKER_SIZE = 40
DASH = (0, (5, 3))

ann_to_stage = {
    "sleep stage w": "W",
    "sleep stage n1": "N1", "sleep stage 1": "N1",
    "sleep stage n2": "N2", "sleep stage 2": "N2",
    "sleep stage n3": "N3", "sleep stage 3": "N3",
    "sleep stage r": "R",
}


def load_full_night(subject):
    ann = mne.read_annotations(str(FULL_NIGHT_DIR / f"{subject}_sleepscoring.edf"))

    items = []
    for onset, dur, desc in zip(ann.onset, ann.duration, ann.description):
        stage = ann_to_stage.get(str(desc).strip().lower())
        if stage is None:
            continue
        items.append((int(round(onset / EPOCH_SEC)),
                      max(1, int(round(dur / EPOCH_SEC))),
                      stage_y[stage]))

    levels = np.full(max(s + n for s, n, _ in items), np.nan)
    for s, n, level in items:
        levels[s:s + n] = level
    return levels


def best_offset(levels, sub, candidates=range(-3, 4)):
    best = (0, -1.0)
    for off in candidates:
        same = total = 0
        for r in sub.itertuples():
            for e in range(int(r.epoch_start), int(r.epoch_end) + 1):
                if 0 <= e + off < len(levels):
                    total += 1
                    same += int(levels[e + off] == stage_y[r.actual])
        agree = same / total if total else 0.0
        if agree > best[1]:
            best = (off, agree)
    return best


def shift(levels, off):
    if off > 0:
        return levels[off:]
    if off < 0:
        return np.concatenate([np.full(-off, np.nan), levels])
    return levels


def fill_gaps(values):
    filled = values.copy()
    valid = np.where(~np.isnan(filled))[0]
    if len(valid) == 0:
        return filled
    filled[:valid[0]] = filled[valid[0]]
    for i in range(valid[0] + 1, len(filled)):
        if np.isnan(filled[i]):
            filled[i] = filled[i - 1]
    return filled


def build_three_lines(subject, sub):
    full_gt = load_full_night(subject)
    off, agree = best_offset(full_gt, sub)
    full_gt = shift(full_gt, off)
    n_sel_seq = len(sub)
    n_sel_ep = int((sub["epoch_end"] - sub["epoch_start"] + 1).sum())
    print(f"{subject}: epochs = {n_sel_ep} | sequence = {n_sel_seq}")

    n_ep = max(len(full_gt), int(sub["epoch_end"].max()) + 1)
    full_gt = np.concatenate([full_gt, np.full(n_ep - len(full_gt), np.nan)])

    sel_gt = np.full(n_ep, np.nan)
    pred = np.full(n_ep, np.nan)
    ok = np.full(n_ep, -1, dtype=int)

    for r in sub.itertuples():
        sl = slice(int(r.epoch_start), int(r.epoch_end) + 1)
        sel_gt[sl] = stage_y[r.actual]
        pred[sl] = stage_y[r.predicted]
        ok[sl] = int(r.correct)
    return full_gt, sel_gt, pred, ok


for subject in test_table["subject"].unique():
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start").copy()
    full_gt, sel_gt, pred, ok = build_three_lines(subject, sub)
    n_ep = len(full_gt)

    x_start = np.arange(n_ep) * EPOCH_SEC / 3600
    x_end = (np.arange(n_ep) + 1) * EPOCH_SEC / 3600
    x_step = np.append(x_start, x_end[-1])
    full_step = np.append(full_gt, full_gt[-1])

    cx = (sub["epoch_start"].to_numpy() + sub["epoch_end"].to_numpy() + 1) / 2 * EPOCH_SEC / 3600
    sx0 = sub["epoch_start"].to_numpy() * EPOCH_SEC / 3600
    sx1 = (sub["epoch_end"].to_numpy() + 1) * EPOCH_SEC / 3600
    gy = sub["actual"].map(stage_y).to_numpy()
    py = sub["predicted"].map(stage_y).to_numpy()
    correct = sub["correct"].to_numpy()
    ok_color = np.where(correct, "green", "red")

    fig, axes = plt.subplots(3, 1, figsize=(FIG_W, FIG_H), sharex=True)

    axes[0].step(x_step, full_step, where="post", color="black", linewidth=LINE_W)

    axes[1].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[1].hlines(gy, sx0, sx1, color=GT_COLOR, linewidth=BLOCK_W, zorder=2)
    axes[1].scatter(cx, gy, s=MARKER_SIZE, color=GT_COLOR,
                    edgecolor="white", linewidth=1, zorder=3)

    axes[2].step(x_step, full_step, where="post", color="#9e9e9e", linewidth=LINE_W, zorder=1)
    axes[2].hlines(py, sx0, sx1, color=ok_color, linewidth=BLOCK_W, zorder=2)
    axes[2].scatter(cx, py, s=MARKER_SIZE, color=ok_color,
                    edgecolor="white", linewidth=1, zorder=3)

    for ax in axes[1:]:
        for x in cx:
            ax.axvline(x, color=GT_COLOR, linestyle="--", linewidth=0.8, alpha=0.4, zorder=0)

    axes[0].set_title(f"Real all night | {subject}")
    axes[1].set_title(f"Ground Truth (sequences used) | {subject}")
    axes[2].set_title(
        f"Predicted | {subject} | Correct {int(correct.sum())}/{len(sub)} "
        f"({correct.mean():.1%})"
    )

    for ax in axes:
        ax.set_ylabel("Sleep stage")
        ax.set_yticks(list(stage_y.values()))
        ax.set_yticklabels(list(stage_y.keys()))
        ax.set_ylim(-0.5, 4.5)
        ax.xaxis.set_major_locator(MultipleLocator(0.5))
        ax.grid(True, alpha=0.3)

    axes[0].set_xlim(-0.05, x_end[-1] * 1.12)

    axes[1].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color=GT_COLOR, lw=4, label="Ground Truth (selected sequence)"),
    ], loc="lower right", framealpha=0.95)
    axes[2].set_xlabel("Time since recording start (hours)")
    axes[2].legend(handles=[
        Line2D([0], [0], color="#9e9e9e", lw=LINE_W, label="Real all night"),
        Line2D([0], [0], color="green", lw=4, label="Predicted: Correct"),
        Line2D([0], [0], color="red", lw=4, label="Predicted: Incorrect"),
    ], loc="lower right", framealpha=0.95)

    plt.tight_layout()
    plt.savefig(hyp_dir / f"hypnogram_full_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 9.1 สัญญาณดิบทั้ง subject เทียบระยะจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1


def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"


stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

test_subject_list = [
    "SN010", "SN013", "SN020", "SN021", "SN024", "SN028", "SN031", "SN038",
    "SN047", "SN058", "SN067", "SN079", "SN084", "SN085", "SN087", "SN101",
    "SN114", "SN117", "SN136", "SN137", "SN141", "SN146", "SN147",
]
available = set(test_table["subject"])
plot_subjects = [s for s in test_subject_list if s in available]
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")
signal_dir = RESULT_DIR / "signal_whole_recording"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN

for subject in plot_subjects:
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep]

    xs, los, his = [], [], []
    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
        xs.append(np.append(t0 + (np.arange(n_bin) + 0.5) * BIN / FS / 3600, np.nan))
        los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
        his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
    x = np.concatenate(xs)
    lo = np.concatenate(los)
    hi = np.concatenate(his)

    fig, axes = plt.subplots(
        n_sig + 3, 1, figsize=(20, 1.2 * n_sig + 3.5), sharex=True,
        gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
    )

    for k in range(n_sig):
        axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
        lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
        axes[k].set_ylim(-lim, lim)
        axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
        axes[k].grid(alpha=0.3)

    for i, r in sub.iterrows():
        t0 = r["epoch_start"] * 30 / 3600
        t1 = (r["epoch_end"] + 1) * 30 / 3600
        axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
        axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
        axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
        for k in range(n_sig):
            axes[k].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                            alpha=0.1, linewidth=0, zorder=0)

    for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
        ax.set_yticks([])
        ax.set_ylabel(name, rotation=0, ha="right", va="center")

    axes[-1].set_xlim(sub["epoch_start"].min() * 30 / 3600, (sub["epoch_end"].max() + 1) * 30 / 3600)
    axes[-1].set_xlabel("Time since recording start (hours)")

    axes[0].legend(handles=[
        Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
    ] + [
        Line2D([0], [0], color="green", lw=8, label="Correct"),
        Line2D([0], [0], color="red", lw=8, label="Wrong"),
    ], loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8)

    n_ok = int(sub["correct"].sum())
    fig.suptitle(
        f"{subject} | whole recording | correct {n_ok}/{len(sub)} sequences ({n_ok / len(sub):.1%}) "
        f"(blank = no sequence)"
    )
    plt.tight_layout()
    plt.savefig(signal_dir / f"signal_whole_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()

## 9.2 สัญญาณของแต่ละ sequence เทียบระยะการนอนจริงกับที่โมเดลทำนาย (Test, แยกรายคน)

In [ ]:
SEQ_PER_FIG = 20
MODE = "pages"
PLOT_CHANNELS = SELECTED_CHANNELS
BIN = 1
FIG_W = 24
GAP = 0.35


def channel_type(name):
    if name == "Chin":
        return "EMG"
    if name[0] == "E" and name[1].isdigit():
        return "EOG"
    return "EEG"


stage_color = {"W": "#ffd6a5", "N1": "#fdffb6", "N2": "#a0c4ff", "N3": "#bdb2ff", "R": "#ffc6ff"}

test_subject_list = [
    "SN010", "SN013", "SN020", "SN021", "SN024", "SN028", "SN031", "SN038",
    "SN047", "SN058", "SN067", "SN079", "SN084", "SN085", "SN087", "SN101",
    "SN114", "SN117", "SN136", "SN137", "SN141", "SN146", "SN147",
]
available = set(test_table["subject"])
plot_subjects = [s for s in test_subject_list if s in available]
print(f"วาด {len(plot_subjects)} subjects จาก Test set (BIN={BIN})")
signal_dir = RESULT_DIR / "signal_sequences"
signal_dir.mkdir(parents=True, exist_ok=True)

X_all = DATA["test"]["X"]
ch_idx = [SELECTED_CHANNELS.index(c) for c in PLOT_CHANNELS]
n_sig = len(ch_idx)
n_bin = X_all.shape[1] // BIN
seq_sec = X_all.shape[1] / FS

for subject in plot_subjects:
    sub = test_table[test_table["subject"] == subject].sort_values("epoch_start")

    keep, last_end = [], -1
    for i, r in sub.iterrows():
        if r["epoch_start"] > last_end:
            keep.append(i)
            last_end = r["epoch_end"]
    sub = sub.loc[keep]

    all_rows = list(sub.iterrows())
    n_total = len(all_rows)
    n_ok_all = int(sub["correct"].sum())

    if MODE == "even" and n_total > SEQ_PER_FIG:
        pick = np.linspace(0, n_total - 1, SEQ_PER_FIG).round().astype(int)
        pages = [[all_rows[j] for j in pick]]
    else:
        pages = [all_rows[p:p + SEQ_PER_FIG] for p in range(0, n_total, SEQ_PER_FIG)]

    for page_no, rows in enumerate(pages, 1):
        n_seq = len(rows)

        xs, los, his = [], [], []
        for k, (i, r) in enumerate(rows):
            x0 = k * (1 + GAP)
            seg = np.asarray(X_all[i][: n_bin * BIN, :])[:, ch_idx].reshape(n_bin, BIN, n_sig)
            rel = (np.arange(n_bin) + 0.5) * BIN / FS / seq_sec
            xs.append(np.append(x0 + rel, np.nan))
            los.append(np.vstack([seg.min(axis=1), np.full((1, n_sig), np.nan)]))
            his.append(np.vstack([seg.max(axis=1), np.full((1, n_sig), np.nan)]))
        x = np.concatenate(xs)
        lo = np.concatenate(los)
        hi = np.concatenate(his)

        fig, axes = plt.subplots(
            n_sig + 3, 1, figsize=(FIG_W, 1.2 * n_sig + 3.5), sharex=True,
            gridspec_kw={"height_ratios": [2] * n_sig + [1, 1, 0.6]}
        )

        for k in range(n_sig):
            # BIN = 1 ทำให้ min == max จึงวาดเป็นเส้น (fill_between จะไม่เห็นแถบ)
            axes[k].plot(x, hi[:, k], color="blue", linewidth=0.3)
            lim = np.nanpercentile(np.abs(np.concatenate([lo[:, k], hi[:, k]])), 99.5)
            axes[k].set_ylim(-lim, lim)
            axes[k].set_ylabel(f"{PLOT_CHANNELS[k]}\n[{channel_type(PLOT_CHANNELS[k])}] microvolt", fontsize=8)
            axes[k].grid(alpha=0.3)

        centers, labels = [], []
        for k, (i, r) in enumerate(rows):
            t0, t1 = k * (1 + GAP), k * (1 + GAP) + 1
            axes[-3].axvspan(t0, t1, facecolor=stage_color[r["actual"]], linewidth=0)
            axes[-2].axvspan(t0, t1, facecolor=stage_color[r["predicted"]], linewidth=0)
            axes[-1].axvspan(t0, t1, facecolor="green" if r["correct"] else "red", linewidth=0)
            for kk in range(n_sig):
                axes[kk].axvspan(t0, t1, facecolor="green" if r["correct"] else "red",
                                 alpha=0.1, linewidth=0, zorder=0)

            axes[-3].text((t0 + t1) / 2, 0.5, r["actual"], ha="center", va="center",
                          fontsize=10, fontweight="bold", transform=axes[-3].get_xaxis_transform())
            axes[-2].text((t0 + t1) / 2, 0.5, r["predicted"], ha="center", va="center",
                          fontsize=10, fontweight="bold", transform=axes[-2].get_xaxis_transform())
            centers.append((t0 + t1) / 2)
            labels.append(f"{r['epoch_start'] * 30 / 3600:.2f} h\n(epoch {int(r['epoch_start'])}-{int(r['epoch_end'])})")

        for k in range(n_seq - 1):
            xg = k * (1 + GAP) + 1 + GAP / 2
            for ax in axes:
                ax.axvline(xg, color="gray", linestyle=":", linewidth=1)

        for ax, name in [(axes[-3], "Ground Truth"), (axes[-2], "Predicted"), (axes[-1], "Correct?")]:
            ax.set_yticks([])
            ax.set_ylabel(name, rotation=0, ha="right", va="center")

        n_slot = max(n_seq, SEQ_PER_FIG)   # ภาพสุดท้ายเว้นที่ว่างไว้ ให้ขนาดบล็อกเท่ากับภาพอื่น
        axes[-1].set_xlim(-GAP / 2, n_slot * (1 + GAP) - GAP / 2)
        axes[-1].set_xticks(centers)
        axes[-1].set_xticklabels(labels, fontsize=8)
        axes[-1].set_xlabel("Selected sequences in time order (gaps removed; tick = real time since recording start)")

        axes[0].legend(handles=[
            Line2D([0], [0], color=stage_color[s], lw=8, label=s) for s in STAGES
        ] + [
            Line2D([0], [0], color="green", lw=8, label="Correct"),
            Line2D([0], [0], color="red", lw=8, label="Wrong"),
        ], loc="lower right", bbox_to_anchor=(1, 1.01), ncol=7, fontsize=8)

        n_ok_shown = sum(bool(r["correct"]) for _, r in rows)
        if MODE == "pages":
            first = (page_no - 1) * SEQ_PER_FIG + 1
            page_txt = f" | page {page_no}/{len(pages)} (sequences {first}-{first + n_seq - 1} of {n_total})"
        else:
            page_txt = ""
        fig.suptitle(
            f"{subject}{page_txt} | shown {n_ok_shown}/{n_seq} correct | "
            f"whole subject {n_ok_all}/{n_total} correct ({n_ok_all / n_total:.1%})"
        )
        plt.tight_layout()
        suffix = "" if MODE == "even" else f"_p{page_no}"
        plt.savefig(signal_dir / f"signal_sequences_{subject}{suffix}.png", dpi=150, bbox_inches="tight")
        plt.show()
        plt.close()

## 10. ตารางภาพรวมทุก subject: ทายถูก/ทั้งหมด ในแต่ละระยะจริง (Test)

In [ ]:
total = test_table.groupby(["subject", "actual"]).size().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0)
right = test_table.groupby(["subject", "actual"])["correct"].sum().unstack(fill_value=0).reindex(columns=STAGES, fill_value=0).astype(int)
rate = right / total.where(total > 0, 1)

order = rate.mean(axis=1).sort_values(ascending=False, kind="stable").index
total, right, rate = total.loc[order], right.loc[order], rate.loc[order]
n = len(rate)

green = np.array([0.78, 0.92, 0.78])
red = np.array([0.96, 0.70, 0.70])
colors = red + (green - red) * rate.to_numpy()[:, :, None]

fig, ax = plt.subplots(figsize=(8, 0.4 * n + 2))
ax.imshow(colors, aspect="auto")

for i in range(n):
    for j in range(len(STAGES)):
        ax.text(j, i, f"{right.iloc[i, j]}/{total.iloc[i, j]}", ha="center", va="center", fontsize=9)

ax.set_xticks(range(len(STAGES)))
ax.set_xticklabels([f"Actual {s}\n{right[s].sum()}/{total[s].sum()}" for s in STAGES])
ax.xaxis.tick_top()
ax.set_yticks(range(n))
ax.set_yticklabels([f"{s} ({right.loc[s].sum()}/{total.loc[s].sum()})" for s in rate.index], fontsize=8)
ax.set_title("Test seed 43: correct/total sequences per actual stage (green = high, red = low)", pad=40)
plt.tight_layout()
plt.savefig(RESULT_DIR / "proof_table_test.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. รายละเอียดรายคน: สัญญาณ 30 วินาที, ความน่าจะเป็นของโมเดล (Test)

In [ ]:
tables = {"test": test_table}


def visualize_subject(split, subject):
    meta = DATA[split]["meta"]
    idx = []
    for label in LABELS:
        rows = meta.index[(meta["subject"] == subject) & (meta["label"] == label)].to_numpy()
        idx.append(rows[0])
    table = tables[split]
    t = np.arange(EPOCH_LEN) / FS

    fig, axes = plt.subplots(N_CHANNELS + 1, len(idx), figsize=(3.3 * len(idx), 2 * (N_CHANNELS + 1)), squeeze=False)

    for col, i in enumerate(idx):
        row = table.loc[i]
        signal = DATA[split]["X"][i][-EPOCH_LEN:, :]
        color = "green" if row["correct"] else "red"

        axes[0, col].set_title(f"Expert: {row['actual']}\nModel: {row['predicted']}", color=color, fontweight="bold")

        for c in range(N_CHANNELS):
            axes[c, col].plot(t, signal[:, c], lw=0.6)
            axes[c, col].set_xlim(0, 30)
            axes[c, col].grid(alpha=0.3)

        probs = [row["prob_" + s] for s in STAGES]
        axes[-1, col].bar(STAGES, probs, color="lightgray")
        axes[-1, col].bar(row["predicted"], row["prob_" + row["predicted"]], color=color)
        axes[-1, col].set_ylim(0, 1)
        axes[-1, col].set_xlabel("Probability")

    for c in range(N_CHANNELS):
        low = min(axes[c, col].get_ylim()[0] for col in range(len(idx)))
        high = max(axes[c, col].get_ylim()[1] for col in range(len(idx)))
        for col in range(len(idx)):
            axes[c, col].set_ylim(low, high)
        axes[c, 0].set_ylabel(SELECTED_CHANNELS[c])

    plt.suptitle(f"{split} | {subject} | raw signal (microvolt) of the last 30 s epoch, first sequence of each stage")
    plt.tight_layout()

    out_dir = RESULT_DIR / "visualize"
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / f"{split}_{subject}.png", dpi=150, bbox_inches="tight")
    plt.show()


subject_acc = test_table.groupby("subject")["correct"].mean().sort_values()
picks = np.linspace(0, len(subject_acc) - 1, min(5, len(subject_acc)), dtype=int)
selected_subjects = subject_acc.index[picks].tolist()
display(subject_acc.loc[selected_subjects])

for subject in selected_subjects:
    visualize_subject("test", subject)

## 12. แถวสรุปผล

In [ ]:
summary_row = pd.DataFrame([{
    "experiment": "exp8_cnn_7ch_class_weighted",
    "seed": SEED,
    "n_test_subjects": len(test_subjects),
    "n_test_sequences": len(y_test),
    "test_accuracy": round(overall["accuracy"], 4),
    "test_macro_f1": round(overall["macro_f1"], 4),
    "test_kappa": round(overall["cohen_kappa"], 4),
}])
display(summary_row)
summary_row.to_csv(RESULT_DIR / "test_experiment_row.csv", index=False)